# EEG2DMN_NestedRegularization_v2
**One predefined round of regularization tuning, chosen without ever looking at the test subject.**

v1 showed that the deep models overfit the training subjects within 1–2 epochs and that Ridge
preferred the strongest penalty it was offered. v2 tests whether stronger regularization helps:

* **Transformer + SupCon** and **Transformer (no SupCon)**: 6 fixed candidate settings each.
* **MLP**: 6 fixed candidate settings (same tuning budget as the Transformers).
* **Ridge**: penalty grid extended from 1e-1…1e5 to 1e-1…1e9.
* **HRF-alpha** and **LaBraM (scratch)** are not retrained; their v1 runs used the same data and
  splits and are read from the v1 output folder for the comparison table.

**Nested selection.** In every LOSO fold and seed, each candidate is trained on 18 subjects and
scored on the 3 validation subjects (mean per-subject Pearson r). Only the best-scoring candidate
is applied to the test subject. Test predictions of the other candidates are never computed.

No preprocessing is repeated: the EEG/fMRI cache written by v1 is reused.
**Run:** Runtime → Run all. The notebook resumes after a disconnect.

## Cell 1: Settings
Normally nothing needs changing. `SMOKE_TEST = True` is only for a quick code check.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

SMOKE_TEST = False

V1_DIR = ""     # v1 output folder (contains cache/ and runs/); "" = auto-detect
OUT_DIR = ""    # "" = <parent of V1_DIR>/Outputs_v2

SEEDS = [0, 1, 2]
TR, WINDOW, LAG_TRS, N_BINS = 2.1, 10, 3, 5
BATCH, MAX_EPOCHS, PATIENCE, N_VAL_SUBJECTS = 64, 50, 10, 3
N_HEADS, D_FF_MULT = 4, 2
LAMBDA_SUPCON, TAU = 0.5, 0.1
MIN_EXACT_WINDOWS = 20

# Candidate settings, fixed before the run. T1 / M1 are the v1 settings.
TRANSFORMER_GRID = {
    "T1": dict(d_model=128, n_layers=4, dropout=0.1, wd=1e-4, lr=1e-3, noise=0.0),
    "T2": dict(d_model=128, n_layers=4, dropout=0.3, wd=1e-2, lr=1e-3, noise=0.0),
    "T3": dict(d_model=64, n_layers=2, dropout=0.3, wd=1e-2, lr=1e-3, noise=0.0),
    "T4": dict(d_model=64, n_layers=2, dropout=0.5, wd=1e-2, lr=3e-4, noise=0.0),
    "T5": dict(d_model=64, n_layers=2, dropout=0.3, wd=1e-2, lr=3e-4, noise=0.5),
    "T6": dict(d_model=32, n_layers=1, dropout=0.5, wd=5e-2, lr=3e-4, noise=0.5),
}
MLP_GRID = {
    "M1": dict(hidden=(256, 128), dropout=0.2, wd=1e-4, lr=1e-3, noise=0.0),
    "M2": dict(hidden=(256, 128), dropout=0.5, wd=1e-2, lr=1e-3, noise=0.0),
    "M3": dict(hidden=(64, 32), dropout=0.5, wd=1e-2, lr=1e-3, noise=0.0),
    "M4": dict(hidden=(64, 32), dropout=0.5, wd=1e-2, lr=3e-4, noise=0.0),
    "M5": dict(hidden=(64, 32), dropout=0.5, wd=1e-2, lr=3e-4, noise=0.5),
    "M6": dict(hidden=(32, 16), dropout=0.5, wd=5e-2, lr=3e-4, noise=0.5),
}
RIDGE_ALPHAS = [10.0 ** k for k in range(-1, 10)]
MODELS = ["ridge", "mlp", "transformer_nocon", "transformer_supcon"]
GRIDS = {"mlp": MLP_GRID, "transformer_nocon": TRANSFORMER_GRID, "transformer_supcon": TRANSFORMER_GRID}

if SMOKE_TEST:
    SEEDS, MAX_EPOCHS = [0], 2
print("Settings loaded. SMOKE_TEST =", SMOKE_TEST)

## Cell 2: Connect Google Drive (Colab only)

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
print("Colab:", IN_COLAB)

## Cell 3: Imports, reproducibility, folders, environment log

In [ ]:
import glob, json, time, copy, random, platform, warnings
import numpy as np
import pandas as pd
import scipy, scipy.stats
import sklearn
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_v1_dir():
    """Folder holding cache/sub-01_ses-01.npz written by EEG2DMN_UnifiedLOSO_v1."""
    bases = ["/content/drive/MyDrive/Research_Dataset", "/content/drive/MyDrive", "/kaggle/working",
             "/kaggle/input", os.getcwd()]
    for base in bases:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if root[len(base):].count(os.sep) >= 5:
                dirs[:] = []
                continue
            if os.path.basename(root) == "cache" and "sub-01_ses-01.npz" in files \
                    and os.path.isdir(os.path.join(os.path.dirname(root), "runs")):
                return os.path.dirname(root)
    raise FileNotFoundError("v1 outputs not found. Set V1_DIR in Cell 1 to the folder that "
                            "contains cache/ and runs/.")


V1_DIR = V1_DIR or find_v1_dir()
CACHE_DIR = os.path.join(V1_DIR, "cache")
V1_RUNS = os.path.join(V1_DIR, "runs")
if not OUT_DIR:
    OUT_DIR = os.path.join(os.path.dirname(os.path.abspath(V1_DIR)),
                           "Outputs_v2" + ("_SMOKE" if SMOKE_TEST else ""))
RUNS_DIR, RES_DIR, FIG_DIR = (os.path.join(OUT_DIR, d) for d in ("runs", "results", "figures"))
for d in (RUNS_DIR, RES_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

env = {"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "cuda": torch.version.cuda,
       "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "v1_dir": V1_DIR, "out_dir": OUT_DIR, "smoke_test": SMOKE_TEST}
with open(os.path.join(RES_DIR, "environment.json"), "w") as fh:
    json.dump(env, fh, indent=2)
with open(os.path.join(RES_DIR, "candidate_grids.json"), "w") as fh:
    json.dump({"transformer": TRANSFORMER_GRID, "mlp": MLP_GRID, "ridge_alphas": RIDGE_ALPHAS}, fh, indent=2)
print(json.dumps(env, indent=2))

## Cell 4: Rebuild the v1 windows from the cache
Same rule as v1: 10 consecutive good EEG volumes, target = DMN 3 TRs after the last one.
The window count must equal v1's (printed in v1 Cell 6).

In [ ]:
cache_files = sorted(glob.glob(os.path.join(CACHE_DIR, "sub-*_ses-*.npz")))
if not cache_files:
    raise FileNotFoundError(f"No cache files in {CACHE_DIR}")
X_list, y_list, sub_list, ses_list, ex_list = [], [], [], [], []
for f in cache_files:
    sub, ses = os.path.basename(f)[:-4].split("_")
    d = np.load(f)
    vols, exact, feats, dmn = d["vols"], d["exact"], d["feats"], d["dmn"]
    pos = {int(v): i for i, v in enumerate(vols)}
    for v in vols:
        v = int(v)
        idx = [pos.get(v - j) for j in range(WINDOW - 1, -1, -1)]
        if any(i is None for i in idx) or v + LAG_TRS >= len(dmn):
            continue
        X_list.append(feats[idx].reshape(WINDOW, -1))
        y_list.append(dmn[v + LAG_TRS])
        sub_list.append(sub)
        ses_list.append(ses)
        ex_list.append(bool(exact[idx].all()))

X_ALL = np.asarray(X_list, dtype=np.float32)
Y_ALL = np.asarray(y_list, dtype=np.float32)
SUB_ALL = np.asarray(sub_list)
SES_ALL = np.asarray(ses_list)
EX_ALL = np.asarray(ex_list, dtype=bool)
SUBJECTS = sorted(set(sub_list))
N_INPUT = X_ALL.shape[2]
v1_summary_path = os.path.join(V1_DIR, "results", "data_summary.json")
if os.path.isfile(v1_summary_path):
    v1n = json.load(open(v1_summary_path))["n_windows_total"]
    if v1n != len(Y_ALL):
        raise RuntimeError(f"Window count {len(Y_ALL)} differs from v1 ({v1n}). Wrong cache folder?")
print(f"Sessions: {len(cache_files)}  subjects: {len(SUBJECTS)}  windows: {len(Y_ALL)} "
      f"(exact {EX_ALL.sum()})  features/volume: {N_INPUT}")

## Cell 5: Models and training code

In [ ]:
class TransformerDMN(nn.Module):
    def __init__(self, n_input, d_model, n_layers, dropout):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(n_input, d_model), nn.LayerNorm(d_model),
                                        nn.GELU(), nn.Dropout(dropout))
        self.pos_embed = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, N_HEADS, D_FF_MULT * d_model, dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.attn_query = nn.Linear(d_model, 1)
        self.proj_head = nn.Sequential(nn.Linear(d_model, 64), nn.BatchNorm1d(64),
                                       nn.ReLU(inplace=True), nn.Linear(64, 32))
        self.reg_head = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout),
                                      nn.Linear(64, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.transformer(self.input_proj(x) + self.pos_embed[:, :x.size(1)])
        w = F.softmax(self.attn_query(h).squeeze(-1), dim=-1).unsqueeze(-1)
        z = (h * w).sum(1)
        return z, self.proj_head(z), self.reg_head(z).squeeze(-1)


class MLP(nn.Module):
    def __init__(self, n_input, hidden, dropout):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(nn.Linear(WINDOW * n_input, h1), nn.ReLU(inplace=True), nn.Dropout(dropout),
                                 nn.Linear(h1, h2), nn.ReLU(inplace=True), nn.Dropout(dropout),
                                 nn.Linear(h2, 1))

    def forward(self, x):
        return None, None, self.net(x.flatten(1)).squeeze(-1)


def supcon_loss(features, labels, tau=TAU):
    b = features.size(0)
    f = F.normalize(features, dim=1)
    sim = f @ f.t() / tau
    eye = torch.eye(b, dtype=torch.bool, device=f.device)
    pos = (labels[:, None] == labels[None, :]) & ~eye
    n_pos = pos.sum(1)
    if (n_pos > 0).sum() == 0:
        return sim.sum() * 0.0
    logits = sim - sim.detach().max(1, keepdim=True).values
    log_den = torch.log((torch.exp(logits) * (~eye)).sum(1, keepdim=True) + 1e-12)
    mean_pos = ((logits - log_den) * pos).sum(1) / n_pos.clamp(min=1)
    return -mean_pos[n_pos > 0].mean()


def build(name, cfg):
    if name == "mlp":
        return MLP(N_INPUT, cfg["hidden"], cfg["dropout"])
    return TransformerDMN(N_INPUT, cfg["d_model"], cfg["n_layers"], cfg["dropout"])


def n_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def r_of(a, b):
    return float(np.corrcoef(a, b)[0, 1]) if len(a) > 2 and a.std() > 1e-12 and b.std() > 1e-12 else np.nan


def mean_subject_r(pred, y, subs):
    """Validation score: mean over subjects of per-subject Pearson r (same metric as the paper)."""
    rs = [r_of(pred[subs == s], y[subs == s]) for s in np.unique(subs)]
    rs = [r for r in rs if np.isfinite(r)]
    return float(np.mean(rs)) if rs else -np.inf


@torch.no_grad()
def predict(model, X):
    model.eval()
    out = []
    for i in range(0, len(X), 256):
        out.append(model(torch.from_numpy(X[i:i + 256]).to(DEVICE))[2].float().cpu().numpy())
    return np.concatenate(out)


def train_candidate(name, cfg, Xtr, ytr, Xva, yva, sva, seed):
    """Train one candidate; early stopping on the validation score. Returns (model, best score, log)."""
    set_seed(seed)
    model = build(name, cfg).to(DEVICE)
    edges = np.quantile(ytr, np.linspace(0, 1, N_BINS + 1))[1:-1]
    bins = np.digitize(ytr, edges).astype(np.int64)
    dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr), torch.from_numpy(bins)),
                    batch_size=BATCH, shuffle=True, drop_last=True,
                    generator=torch.Generator().manual_seed(seed), num_workers=0)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best, best_state, best_ep, bad, log = -np.inf, None, 0, 0, []
    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        tot, n = 0.0, 0
        for xb, yb, bb in dl:
            xb, yb, bb = xb.to(DEVICE), yb.to(DEVICE), bb.to(DEVICE)
            if cfg["noise"] > 0:
                xb = xb + cfg["noise"] * torch.randn_like(xb)
            _, proj, pred = model(xb)
            loss = F.mse_loss(pred, yb)
            if name == "transformer_supcon":
                loss = loss + LAMBDA_SUPCON * supcon_loss(proj, bb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item() * len(xb)
            n += len(xb)
        sched.step()
        pv = predict(model, Xva)
        score = mean_subject_r(pv, yva, sva)
        log.append({"epoch": ep, "train_loss": tot / max(n, 1), "val_mean_r": score,
                    "val_mse": float(np.mean((pv - yva) ** 2))})
        if score > best:
            best, best_ep, bad = score, ep, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best, best_ep, log


param_counts = {f"transformer_{k}": n_params(build("transformer_nocon", c)) for k, c in TRANSFORMER_GRID.items()}
param_counts.update({f"mlp_{k}": n_params(build("mlp", c)) for k, c in MLP_GRID.items()})
with open(os.path.join(RES_DIR, "parameter_counts.json"), "w") as fh:
    json.dump(param_counts, fh, indent=2)
print("Trainable parameters per candidate:", param_counts)

## Cell 6: Nested LOSO (resumable)
For each model, seed and test subject: train all candidates on 18 subjects, score them on the
3 validation subjects (the same split as v1), apply only the winner to the test subject.

In [ ]:
folds = SUBJECTS[:2] if SMOKE_TEST else SUBJECTS
t_start = time.time()
for name in MODELS:
    for seed in ([0] if name == "ridge" else SEEDS):
        run_dir = os.path.join(RUNS_DIR, name, f"seed{seed}")
        os.makedirs(run_dir, exist_ok=True)
        for fi, test_sub in enumerate(folds):
            out = os.path.join(run_dir, f"fold{fi:02d}_{test_sub}.npz")
            if os.path.isfile(out):
                continue
            t0 = time.time()
            te = SUB_ALL == test_sub
            train_subs = [s for s in SUBJECTS if s != test_sub]
            rng = np.random.RandomState(10_000 + 100 * seed + fi)
            val_subs = sorted(rng.choice(train_subs, N_VAL_SUBJECTS, replace=False).tolist())
            info = {"model": name, "seed": seed, "fold": fi, "test_subject": test_sub}
            if name == "ridge":
                tr = ~te
                gs = GridSearchCV(make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": RIDGE_ALPHAS},
                                  cv=GroupKFold(n_splits=5), scoring="neg_mean_squared_error")
                gs.fit(X_ALL[tr].reshape(tr.sum(), -1), Y_ALL[tr], groups=SUB_ALL[tr])
                y_pred = gs.predict(X_ALL[te].reshape(te.sum(), -1))
                alpha = float(gs.best_params_["ridge__alpha"])
                info.update({"selected": f"alpha={alpha:g}", "alpha": alpha,
                             "alpha_at_grid_edge": alpha in (RIDGE_ALPHAS[0], RIDGE_ALPHAS[-1]),
                             "n_train": int(tr.sum()), "n_test": int(te.sum())})
            else:
                va = np.isin(SUB_ALL, val_subs)
                tr = ~te & ~va
                cands, best_model = {}, None
                for key, cfg in GRIDS[name].items():
                    model, score, best_ep, log = train_candidate(name, cfg, X_ALL[tr], Y_ALL[tr],
                                                                 X_ALL[va], Y_ALL[va], SUB_ALL[va], seed)
                    cands[key] = {"val_mean_r": score, "best_epoch": best_ep, "epochs_run": len(log)}
                    pd.DataFrame(log).to_csv(out.replace(".npz", f"_{key}_log.csv"), index=False)
                    if best_model is None or score > cands[best_model[0]]["val_mean_r"]:
                        best_model = (key, model)
                    else:
                        del model
                key, model = best_model
                y_pred = predict(model, X_ALL[te])
                info.update({"selected": key, "candidates": cands, "val_subjects": val_subs,
                             "best_epoch": cands[key]["best_epoch"], "n_train": int(tr.sum()),
                             "n_val": int(va.sum()), "n_test": int(te.sum())})
            y_true, ex = Y_ALL[te], EX_ALL[te]
            info.update({"pearson_r": r_of(y_pred, y_true), "mse": float(np.mean((y_pred - y_true) ** 2)),
                         "n_test_exact": int(ex.sum()),
                         "pearson_r_exact": r_of(y_pred[ex], y_true[ex]) if ex.sum() >= MIN_EXACT_WINDOWS else None,
                         "seconds": round(time.time() - t0, 1)})
            np.savez_compressed(out, y_true=y_true, y_pred=np.asarray(y_pred, np.float32), exact=ex,
                                session=SES_ALL[te], info=json.dumps(info))
            print(f"{name:19s} seed{seed} fold{fi:02d} {test_sub}: chose {info['selected']:>12s}  "
                  f"r={info['pearson_r']:+.4f}  ({info['seconds']}s, total {(time.time() - t_start) / 60:.1f} min)",
                  flush=True)
print("All runs finished.")

## Cell 7: Results, statistics and figures
Combines the v2 models with the v1 HRF-alpha and LaBraM runs (same data, same splits).
* Per-subject r = mean over seeds; table = mean ± SD over subjects.
* `p_vs_zero`: one-sample t-test of per-subject r against 0.
* `p_vs_ref`: paired tests against `transformer_supcon` (v2).

In [ ]:
def load_runs(root, models, tag):
    rows = []
    for m in models:
        for f in glob.glob(os.path.join(root, m, "seed*", "fold*.npz")):
            info = json.loads(str(np.load(f)["info"]))
            info["model"] = f"{m}{tag}"
            info.pop("candidates", None)
            rows.append(info)
    return rows


rows = load_runs(RUNS_DIR, MODELS, "")
rows += load_runs(V1_RUNS, ["hrf_alpha", "labram_scratch"], "_v1")
runs = pd.DataFrame(rows)
for col in ("pearson_r", "pearson_r_exact", "mse"):
    runs[col] = pd.to_numeric(runs[col], errors="coerce")
runs.loc[runs.model == "hrf_alpha_v1", "mse"] = np.nan        # unscaled prediction; MSE not meaningful
runs.sort_values(["model", "seed", "fold"]).to_csv(os.path.join(RES_DIR, "all_runs.csv"), index=False)

order = [m for m in ["hrf_alpha_v1", "ridge", "mlp", "labram_scratch_v1", "transformer_nocon",
                     "transformer_supcon"] if m in set(runs.model)]
per_sub = runs.groupby(["model", "test_subject"])["pearson_r"].mean().unstack(0)[order]
per_sub_ex = runs.groupby(["model", "test_subject"])["pearson_r_exact"].mean().unstack(0).reindex(columns=order)
per_sub.to_csv(os.path.join(RES_DIR, "per_subject_r.csv"))
per_sub_ex.to_csv(os.path.join(RES_DIR, "per_subject_r_exact_windows.csv"))

REF = "transformer_supcon"
summary, stats = [], {}
for m in order:
    r = per_sub[m].dropna()
    t0_, p0_ = scipy.stats.ttest_1samp(r, 0.0)
    seed_means = runs[runs.model == m].groupby("seed")["pearson_r"].mean()
    row = {"model": m, "n_subjects": len(r), "r_mean": r.mean(), "r_sd": r.std(ddof=1),
           "p_vs_zero": p0_, "n_positive": int((r > 0).sum()),
           "r_mean_exact_windows": per_sub_ex[m].mean(), "n_subjects_exact": int(per_sub_ex[m].notna().sum()),
           "mse_mean": runs[runs.model == m].groupby("test_subject")["mse"].mean().mean(),
           "seed_mean_r_min": seed_means.min(), "seed_mean_r_max": seed_means.max()}
    if m != REF and REF in per_sub:
        both = per_sub[[REF, m]].dropna()
        t, p = scipy.stats.ttest_rel(both[REF], both[m])
        w, pw = scipy.stats.wilcoxon(both[REF] - both[m])
        stats[m] = {"n": len(both), "mean_diff": float((both[REF] - both[m]).mean()), "t": float(t),
                    "p_ttest": float(p), "W": float(w), "p_wilcoxon": float(pw),
                    "n_subjects_ref_better": int((both[REF] > both[m]).sum())}
        row.update({"p_vs_ref_ttest": p, "p_vs_ref_wilcoxon": pw})
    summary.append(row)
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(RES_DIR, "summary_table.csv"), index=False)
with open(os.path.join(RES_DIR, "stats_vs_transformer_supcon.json"), "w") as fh:
    json.dump(stats, fh, indent=2)
pd.set_option("display.width", 220)
print(summary.round(4).to_string(index=False))

# which candidate won, how often
sel = runs[runs.model.isin(MODELS)].groupby(["model", "selected"]).size().rename("times_selected").reset_index()
sel.to_csv(os.path.join(RES_DIR, "selection_counts.csv"), index=False)
print("\nCandidate selected (count over folds x seeds):")
print(sel.to_string(index=False))
ridge_edge = runs[runs.model == "ridge"]["alpha_at_grid_edge"]
print(f"\nRidge alpha at the edge of the grid in {int(ridge_edge.sum())}/{len(ridge_edge)} folds")

# Figures
labels = {"hrf_alpha_v1": "HRF-alpha", "ridge": "Ridge", "mlp": "MLP", "labram_scratch_v1": "LaBraM\n(scratch)",
          "transformer_nocon": "Transformer\n(no SupCon)", "transformer_supcon": "Transformer\n+ SupCon"}
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(order)), summary["r_mean"], yerr=summary["r_sd"], capsize=5, color="#4C72B0")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([labels[m] for m in order])
ax.axhline(0, color="k", lw=0.6)
ax.set_ylabel("Pearson r (mean ± SD over subjects)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "model_comparison.png"), dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 3.5))
im = ax.imshow(per_sub[order].T.values, aspect="auto", cmap="RdYlGn", vmin=-0.3, vmax=0.5)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([labels[m].replace("\n", " ") for m in order])
ax.set_xticks(range(len(per_sub)))
ax.set_xticklabels(per_sub.index, rotation=45, ha="right")
fig.colorbar(im, ax=ax, label="Pearson r")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "per_subject_heatmap.png"), dpi=200)
plt.close(fig)

if {REF, "transformer_nocon"} <= set(per_sub.columns):
    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(len(per_sub))
    ax.bar(x - 0.2, per_sub[REF], 0.4, label="With SupCon")
    ax.bar(x + 0.2, per_sub["transformer_nocon"], 0.4, label="Without SupCon")
    ax.set_xticks(x)
    ax.set_xticklabels(per_sub.index, rotation=45, ha="right")
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel("Pearson r")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "ablation_paired.png"), dpi=200)
    plt.close(fig)

piv = sel.pivot(index="selected", columns="model", values="times_selected").fillna(0)
fig, ax = plt.subplots(figsize=(8, 4))
piv.plot.bar(ax=ax)
ax.set_ylabel("Times selected (folds × seeds)")
ax.set_xlabel("Candidate setting")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "selection_counts.png"), dpi=200)
plt.close(fig)

print("\nFiles written:")
for d in (RES_DIR, FIG_DIR):
    for f in sorted(os.listdir(d)):
        print("  ", os.path.join(os.path.basename(d), f))